# Asteroid-Hazard Capstone — reproducible notebook

Predict NASA's *potentially hazardous* flag for near-Earth objects, worked the honest way.
Every number in the capstone pages comes from running this notebook top to bottom.
Deterministic (random_state=0); needs pandas, numpy, scikit-learn. Data: neo_v2.csv (bundled).


In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix, precision_score, recall_score)

RS = 0
df = pd.read_csv("neo_v2.csv")
df.shape

## 1 · First contact & integrity
What is one row? Objects repeat across close approaches — so we must split by object, not row.


In [ ]:
print("rows / unique ids:", len(df), df.id.nunique())
print("constant cols:", [c for c in df.columns if df[c].nunique()==1])
print("positive rate:", round(df.hazardous.mean(),4))

## 2 · Redundancy
The two diameter columns are a deterministic function of absolute_magnitude (log-corr = -1).


In [ ]:
print("ratio max/min:", (df.est_diameter_max/df.est_diameter_min).mean(), "(= sqrt5)")
print("log-corr:", round(np.corrcoef(np.log(df.est_diameter_min), df.absolute_magnitude)[0,1],4))

## 3 · The harness: metric (PR-AUC) + honest split (grouped)
A random split lets an object leak across train/test and inflates flexible models. Grouped-by-id is honest.


In [ ]:
FEATURES=["absolute_magnitude","relative_velocity","miss_distance"]
X=df[FEATURES].to_numpy(); y=df.hazardous.astype(int).to_numpy(); groups=df.id.to_numpy()

def bench(Xtr,Xte,ytr,yte):
    out={}
    def row(name,s): out[name]=(roc_auc_score(yte,s),average_precision_score(yte,s))
    sc=StandardScaler().fit(Xtr)
    row("logistic",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=RS).fit(sc.transform(Xtr),ytr).predict_proba(sc.transform(Xte))[:,1])
    row("tree_d6",DecisionTreeClassifier(max_depth=6,class_weight="balanced",random_state=RS).fit(Xtr,ytr).predict_proba(Xte)[:,1])
    row("forest",RandomForestClassifier(n_estimators=300,class_weight="balanced",n_jobs=-1,random_state=RS).fit(Xtr,ytr).predict_proba(Xte)[:,1])
    row("histgb",HistGradientBoostingClassifier(random_state=RS).fit(Xtr,ytr).predict_proba(Xte)[:,1])
    return out

Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=RS)
rand=bench(Xtr,Xte,ytr,yte)
tr,te=next(GroupShuffleSplit(1,test_size=.25,random_state=RS).split(X,y,groups))
grp=bench(X[tr],X[te],y[tr],y[te])
print("model         RANDOM_PR  GROUPED_PR")
for k in grp: print(f"{k:12} {rand[k][1]:.3f}     {grp[k][1]:.3f}")

The forest inflates most under the random split (0.566 vs honest 0.478) — that gap is object leakage.


## 4 · The models on the honest harness


In [ ]:
for k,(r,p) in grp.items(): print(f"{k:12} ROC {r:.3f}  PR {p:.3f}")

## 5 · Interpretation — permutation importance (unbiased)
Resolves impurity-importance bias: miss_distance >> velocity, agreeing with the logistic coefficients.


In [ ]:
rf=RandomForestClassifier(n_estimators=300,class_weight="balanced",n_jobs=-1,random_state=RS).fit(X[tr],y[tr])
pi=permutation_importance(rf,X[te],y[te],scoring="average_precision",n_repeats=10,random_state=RS,n_jobs=-1)
for f,m in sorted(zip(FEATURES,pi.importances_mean),key=lambda z:-z[1]): print(f"{f:20} -{m:.3f}")

## 6 · Operating point
A threshold is a cost decision. Missed hazard >> false alarm, so choose high recall.


In [ ]:
p=rf.predict_proba(X[te])[:,1]; yt=y[te]
prec,rec,thr=precision_recall_curve(yt,p)
i=int(np.max(np.where(rec[:-1]>=0.90)))
pred=(p>=thr[i]).astype(int); tn,fp,fn,tp=confusion_matrix(yt,pred).ravel()
print(f"thr {thr[i]:.3f}  recall {recall_score(yt,pred):.3f}  precision {precision_score(yt,pred):.3f}")
print(f"caught {tp}/{tp+fn} hazards, {fp} false alarms")

---
**Verdict:** random forest, PR-AUC 0.478 on the honest grouped split — a genuine ~+0.19 over the size-rule baseline,
at the information ceiling these three features allow (MOID, the orbit half of the label, is missing).
